1. Words talk to each other
   (Attention)

2. Keep original information
   (Residual Connection)

3. Normalize
   (LayerNorm)

4. Process information
   (Feed Forward Network)

5. Keep information again
   (Residual Connection)

6. Normalize again
   (LayerNorm)

In [1]:
# Simple Text Classification with Transformers

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [2]:
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I enjoyed the movie",
    "The film was fantastic",
    "I hate this movie",
    "This film is terrible",
    "Very boring story",
    "Worst acting ever",
    "I disliked the movie",
    "The film was awful"
]

In [3]:
# 1 = positive, 0 = negative

labels = np.array([1, 1, 1, 1, 1, 1,
                   0, 0, 0, 0, 0, 0])

In [4]:
#tokenize and pad sequences
# We set vocab_size to 1000 to keep only the top 1000 most common words in our dataset. 
# The oov_token="<OOV>" argument tells the tokenizer to replace any word not in the top 1000 with a special token "<OOV>" (out-of-vocabulary). 
# This helps handle words that may appear in new data but were not seen during training.

vocab_size = 1000
max_length = 6

# 2. Tokenize and pad sequences
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_length, padding="post")

print("Word Index:")
print(tokenizer.word_index)

print("\nInput Sequences:")
print(X)

Word Index:
{'<OOV>': 1, 'i': 2, 'this': 3, 'movie': 4, 'film': 5, 'the': 6, 'is': 7, 'very': 8, 'acting': 9, 'story': 10, 'was': 11, 'love': 12, 'amazing': 13, 'good': 14, 'excellent': 15, 'enjoyed': 16, 'fantastic': 17, 'hate': 18, 'terrible': 19, 'boring': 20, 'worst': 21, 'ever': 22, 'disliked': 23, 'awful': 24}

Input Sequences:
[[ 2 12  3  4  0  0]
 [ 3  5  7 13  0  0]
 [ 8 14  9  0  0  0]
 [15 10  0  0  0  0]
 [ 2 16  6  4  0  0]
 [ 6  5 11 17  0  0]
 [ 2 18  3  4  0  0]
 [ 3  5  7 19  0  0]
 [ 8 20 10  0  0  0]
 [21  9 22  0  0  0]
 [ 2 23  6  4  0  0]
 [ 6  5 11 24  0  0]]


In [ ]:
# Positional Embeddings in Transformers

# 1. Create a custom layer that combines token and positional embeddings
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [ ]:
# 2. Create a custom Transformer block
# This block includes 
# multi-head self-attention
# feed-forward network
# layer normalization and residual connections.

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes 
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)

        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(out1)

        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

In [7]:
# build the model
embed_dim = 16
num_heads = 2
ff_dim = 32
inputs = layers.Input(shape=(max_length,))

x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)

x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)

x = layers.GlobalAveragePooling1D()(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [8]:
model.compile(

    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 6, 16)          │        16,096 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 6, 16)          │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,409 (75.82 KB)

 Trainable params: 19,409 (75.82 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.fit(
    X,
    labels,
    epochs=30,
    batch_size=2,
    verbose=1
)

Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2500 - loss: 0.9827
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3333 - loss: 0.8464    
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5833 - loss: 0.7044
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6667 - loss: 0.6499    
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.6208 
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8333 - loss: 0.6065 
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8333 - loss: 0.5807
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9167 - loss: 0.5508
Epoch 9/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9167 - loss: 0.5255
Epoch 10/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.4965
Epoch 11/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.4790
Epoch 12/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 0.4603
Epo

In [10]:
test_sentences = [
    "I love the film",
    "This movie was awful"
]

test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)

for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
I love the film -> 0.98789054
Prediction: Positive
This movie was awful -> 0.0101128975
Prediction: Negative


# Modify the previous Transformer code

embed_dim = 16
num_heads = 2
ff_dim = 32
max_length = 8
